In [ ]:
import pandas as pd
import os
import json
curr_wd = os.path.abspath(os.getcwd())
print(curr_wd)
from importlib import reload

In [ ]:
fpath = curr_wd + "/data/input/RG_motif_info_df.parquet"
motif_info_set_df = pd.read_parquet(fpath)
motif_info_set_df

fpath = curr_wd + "/data/input/all_IDR_human.parquet"
IDR_info_df = pd.read_parquet(fpath)
IDR_info_df.rename(columns={"protein_name": "UniqueID"}, inplace=True)
IDR_info_df

# from window_extraction import prepare_window_dataframe
import src.window_extraction as window_extraction
# Reload the module after making changes
reload(window_extraction)

# from fasta_writer import build_fasta_records, write_fasta_chunked

df_windows_7 = window_extraction.prepare_window_dataframe(
    df_motif=motif_info_set_df[motif_info_set_df["Groups_num"] == "7"],
    df_idr=IDR_info_df,
    flank=5,
    mode="adaptive",
    min_idr_fraction=0.9
)
print(df_windows_7.head())
print(df_windows_7.shape)

df_windows_5 = window_extraction.prepare_window_dataframe(
    df_motif=motif_info_set_df[motif_info_set_df["Groups_num"] == "5"],
    df_idr=IDR_info_df,
    flank=5,
    mode="adaptive",
    min_idr_fraction=0.9
)

print(df_windows_5.head())
print(df_windows_5.shape)

df_windows_4 = window_extraction.prepare_window_dataframe(
    df_motif=motif_info_set_df[motif_info_set_df["Groups_num"] == "4"],
    df_idr=IDR_info_df,
    flank=5,
    mode="adaptive",
    min_idr_fraction=0.9
)

print(df_windows_4.head())
print(df_windows_4.shape)

df_windows_6 = window_extraction.prepare_window_dataframe(
    df_motif=motif_info_set_df[motif_info_set_df["Groups_num"] == "6"],
    df_idr=IDR_info_df,
    flank=5,
    mode="adaptive",
    min_idr_fraction=0.9
)

print(df_windows_6.head())
print(df_windows_6.shape)


###### SAVE THE METADATA FOR LATER

path = "/mnt/d/phd/scripts/16_ev_signature_predictor/data/processed"
filename = "group4_RG_regions_win5_metadata.pkl"

df_windows_4.to_pickle(os.path.join(path, filename))

################################################

path = "/mnt/d/phd/scripts/16_ev_signature_predictor/data/processed"
filename = "group5_RG_regions_win5_metadata.pkl"

df_windows_5.to_pickle(os.path.join(path, filename))

################################################

path = "/mnt/d/phd/scripts/16_ev_signature_predictor/data/processed"
filename = "group6_RG_regions_win5_metadata.pkl"

df_windows_6.to_pickle(os.path.join(path, filename))
################################################

path = "/mnt/d/phd/scripts/16_ev_signature_predictor/data/processed"
filename = "group7_RG_regions_win5_metadata.pkl"

df_windows_7.to_pickle(os.path.join(path, filename))


#### GENERATE JSON FILES FOR GNOMAD

result_neg = df_windows_4[["UniqueID", "win_start", "win_end"]].rename(
    columns={"win_start": "start", "win_end": "end"}
).to_dict(orient="records")

path = "/mnt/d/phd/scripts/16_ev_signature_predictor/data/processed"
filename = "group4_RG_regions_win5.json"


with open(f"{path}/{filename}", "w") as f:
    json.dump(result_neg, f, indent=2)

#######################################

result_pos = df_windows_5[["UniqueID", "win_start", "win_end"]].rename(
    columns={"win_start": "start", "win_end": "end"}
).to_dict(orient="records")

path = "/mnt/d/phd/scripts/16_ev_signature_predictor/data/processed"
filename = "group5_RG_regions_win5.json"

with open(f"{path}/{filename}", "w") as f:
    json.dump(result_pos, f, indent=2)

#######################################

result_pos = df_windows_6[["UniqueID", "win_start", "win_end"]].rename(
    columns={"win_start": "start", "win_end": "end"}
).to_dict(orient="records")

path = "/mnt/d/phd/scripts/16_ev_signature_predictor/data/processed"
filename = "group6_RG_regions_win5.json"

with open(f"{path}/{filename}", "w") as f:
    json.dump(result_pos, f, indent=2)


#######################################

result_pos = df_windows_7[["UniqueID", "win_start", "win_end"]].rename(
    columns={"win_start": "start", "win_end": "end"}
).to_dict(orient="records")

path = "/mnt/d/phd/scripts/16_ev_signature_predictor/data/processed"
filename = "group7_RG_regions_win5.json"

with open(f"{path}/{filename}", "w") as f:
    json.dump(result_pos, f, indent=2)

In [ ]:
import os
import json
from importlib import reload
import pandas as pd

import src.window_extraction as window_extraction
reload(window_extraction)

# ── Config ────────────────────────────────────────────────────────────────
GROUPS = ["4", "5", "6", "7"]
FLANK = 5
MIN_IDR_FRACTION = 0.9
MODE = "adaptive"

PROCESSED_DIR = "/mnt/d/phd/scripts/16_ev_signature_predictor/data/processed"
WINDOW_SUFFIX = f"win{FLANK}"

# ── Load inputs ───────────────────────────────────────────────────────────
motif_info_set_df = pd.read_parquet(
    os.path.join(curr_wd, "data/input/RG_motif_info_df.parquet")
)
IDR_info_df = (
    pd.read_parquet(os.path.join(curr_wd, "data/input/all_IDR_human.parquet"))
      .rename(columns={"protein_name": "UniqueID"})
)

# ── Process all groups ────────────────────────────────────────────────────
df_windows_by_group = {}

for group in GROUPS:
    print(f"\n══════════ Group {group} ══════════")
    
    df_windows = window_extraction.prepare_window_dataframe(
        df_motif=motif_info_set_df[motif_info_set_df["Groups_num"] == group],
        df_idr=IDR_info_df,
        flank=FLANK,
        mode=MODE,
        min_idr_fraction=MIN_IDR_FRACTION,
    )
    print(f"  Shape: {df_windows.shape}")
    print(df_windows.head())
    
    df_windows_by_group[group] = df_windows
    
    # Save metadata pickle
    metadata_path = os.path.join(
        PROCESSED_DIR, f"group{group}_RG_regions_{WINDOW_SUFFIX}_metadata.pkl"
    )
    df_windows.to_pickle(metadata_path)
    print(f"  Saved metadata: {metadata_path}")
    
    # Save JSON for gnomAD pipeline
    json_records = (
        df_windows[["UniqueID", "win_start", "win_end"]]
        .rename(columns={"win_start": "start", "win_end": "end"})
        .to_dict(orient="records")
    )
    json_path = os.path.join(
        PROCESSED_DIR, f"group{group}_RG_regions_{WINDOW_SUFFIX}.json"
    )
    with open(json_path, "w") as f:
        json.dump(json_records, f, indent=2)
    print(f"  Saved JSON: {json_path}")

# ── Summary ───────────────────────────────────────────────────────────────
print("\n══════════ Summary ══════════")
for group, df in df_windows_by_group.items():
    print(f"  Group {group}: {len(df):,} windows from "
          f"{df['UniqueID'].nunique():,} proteins")

In [ ]:
import os
import json
from importlib import reload
import pandas as pd
import src.coding_dna as cdna          # your new MANE-based module
import src.build_mane_index as build_mane_index
import src.write_bed_file as write_bed_file
reload(cdna); reload(build_mane_index); reload(write_bed_file)

# ── Config ────────────────────────────────────────────────────────────────
GROUPS = ["4", "5", "6", "7"]
PROCESSED_DIR = "/mnt/d/phd/scripts/16_ev_signature_predictor/data/processed"
WINDOW_SUFFIX = "win5"

# ── Build the MANE CDS index + FASTA ONCE (shared across all groups) ───────
build_mane_index.main()
CDS_INDEX = cdna.load_index()
FASTA = cdna.open_fasta()
cdna.sanity_check_chromosomes(CDS_INDEX, FASTA)

# ── Step 1: gather genomic coordinates per group via the cdna pipeline ─────
genomic_coordinates_by_group = {}
failed_by_group = {}

for group in GROUPS:
    print(f"\n══════════ Group {group}: gathering genomic coordinates ══════════")

    # load the metadata pickle saved by the window-extraction step
    metadata_path = os.path.join(
        PROCESSED_DIR, f"group{group}_RG_regions_{WINDOW_SUFFIX}_metadata.pkl"
    )
    metadata = pd.read_pickle(metadata_path)
    print(f"  Loaded {len(metadata)} regions from {metadata_path}")

    # same call signature as pos/neg
    regions = cdna.df_to_regions(metadata)
    genomic_coords, failed = cdna.process_multiple_regions(regions, CDS_INDEX, FASTA)

    # add region_id (identical to pos/neg)
    for i, el in enumerate(genomic_coords):
        el["region_id"] = f"{el['protein']}_{el['prot_region'][0]}_{el['prot_region'][1]}"
        genomic_coords[i] = el

    genomic_coordinates_by_group[group] = genomic_coords
    failed_by_group[group] = failed
    print(f"  Succeeded: {len(genomic_coords)}, Failed: {len(failed)}")
    if failed:
        print(f"  Failed regions: {failed}")

# ── Step 2: merge all groups into one JSON ────────────────────────────────
merged_genomic_coordinates_list = []
for group in GROUPS:
    merged_genomic_coordinates_list.extend(
        {**d, "group": group} for d in genomic_coordinates_by_group[group]
    )
print(f"\nMerged total: {len(merged_genomic_coordinates_list)} regions "
      f"across {len(GROUPS)} groups")

merged_json_path = os.path.join(
    PROCESSED_DIR, f"genomic_coords_merged_{WINDOW_SUFFIX}_4groups.json"
)
with open(merged_json_path, "w") as fh:
    json.dump(merged_genomic_coordinates_list, fh, indent=4)
print(f"Saved merged JSON: {merged_json_path}")


bed_path = os.path.join(PROCESSED_DIR,
    f"genomic_coords_merged_{WINDOW_SUFFIX}_4groups.bed")
write_bed_file.write_multigroup_bed(
    results_by_group=genomic_coordinates_by_group, output_path=bed_path)

In [ ]:
########################## HERE MOGON RUN ##########################################

In [ ]:
import src.variant_assignment as va

# Load
df_raw = va.load_vep_tsv(
    "/mnt/d/phd/scripts/16_ev_signature_predictor/data/bcftools/combined_vep_variants_4groups.tsv"
)

print("══ Basic structure ══")
print(f"Total rows: {len(df_raw):,}")
print(f"Columns: {df_raw.columns.tolist()}")

print("\n══ Quality distributions ══")
print("\nFILTER:")
print(df_raw["FILTER"].value_counts().head(5))
print("\nBIOTYPE:")
print(df_raw["BIOTYPE"].value_counts().head(5))
print("\nConsequence (top 10):")
print(df_raw["Consequence"].value_counts().head(10))

print("\n══ Chromosome coverage ══")
print("\nRows per chromosome (should match BED counts roughly):")
print(df_raw["CHROM"].value_counts().sort_index())

print("\n══ Identity check: protein coverage ══")
# How many unique UniProt accessions appear in the VEP output?
df_raw["uniprot_short"] = df_raw["UNIPROT_ISOFORM"].apply(
    lambda x: str(x).split("-")[0] if pd.notna(x) else None
)
n_unique_uniprot = df_raw["uniprot_short"].nunique()
n_unique_symbol = df_raw["SYMBOL"].nunique()
print(f"Unique UniProt accessions in VEP output: {n_unique_uniprot}")
print(f"Unique gene symbols in VEP output: {n_unique_symbol}")

print("\n══ Cross-check: how many of our queried proteins were found? ══")
# Load any one of the window metadata pickles to get expected proteins
import os
PROCESSED_DIR = "/mnt/d/phd/scripts/16_ev_signature_predictor/data/processed"
all_expected_proteins = set()
for group in ["4", "5", "6", "7"]:
    df_windows = pd.read_pickle(
        os.path.join(PROCESSED_DIR, f"group{group}_RG_regions_win5_metadata.pkl")
    )
    all_expected_proteins.update(df_windows["UniqueID"].unique())
print(f"Expected proteins (across all 4 groups): {len(all_expected_proteins)}")

found_proteins = set(df_raw["uniprot_short"].dropna().unique())
overlap = all_expected_proteins & found_proteins
missing_from_vep = all_expected_proteins - found_proteins
print(f"Found in VEP output: {len(overlap)} / {len(all_expected_proteins)}")
print(f"Missing from VEP output: {len(missing_from_vep)}")

if len(missing_from_vep) > 0:
    print(f"\nFirst 20 missing proteins:")
    for p in list(missing_from_vep)[:20]:
        print(f"  {p}")

print("\n══ Filtering effect (apply filter) ══")
df_filt = va.filter_vep(df_raw)
print(f"After filtering: {len(df_filt):,} rows ({100*len(df_filt)/len(df_raw):.1f}% kept)")

In [ ]:
# How complete is UniProt in the filtered set?
print("Rows with UNIPROT_ISOFORM populated:", df_filt["UNIPROT_ISOFORM"].notna().sum())
print("Rows without:", df_filt["UNIPROT_ISOFORM"].isna().sum())
print()
print("Sample of populated values:")
print(df_filt["UNIPROT_ISOFORM"].dropna().sample(10).tolist())
print()
print("Gene symbols where UNIPROT_ISOFORM is missing:")
print(df_filt[df_filt["UNIPROT_ISOFORM"].isna()]["SYMBOL"].value_counts().head(10))

import json

with open("/mnt/d/phd/scripts/16_ev_signature_predictor/data/processed/genomic_coords_merged_win5_4groups.json") as f:
    regions = json.load(f)

uniprot_ids = sorted({r["protein"] for r in regions})
print(f"Unique UniProt IDs in regions: {len(uniprot_ids)}")
print(uniprot_ids[:10])

import src.variant_assignment as va

uniprot_to_symbols = va.fetch_uniprot_to_symbols(uniprot_ids, "/mnt/d/phd/scripts/16_ev_signature_predictor/data/processed/uniprot_to_symbols.json")

# Check coverage
print(f"Got mappings for {len(uniprot_to_symbols)} / {len(uniprot_ids)} UniProt IDs")

# Any missing?
missing = set(uniprot_ids) - set(uniprot_to_symbols.keys())
if missing:
    print(f"Missing: {missing}")

# Sanity check — does RBMXL2 appear in any reverse mapping?
for acc, symbols in uniprot_to_symbols.items():
    if "RBMXL2" in symbols:
        print(f"RBMXL2 → UniProt {acc}")
        break


uniprot_lookup, symbol_lookup = va.build_region_lookups(
    "/mnt/d/phd/scripts/16_ev_signature_predictor/data/processed/genomic_coords_merged_win5_4groups.json",
    uniprot_to_symbols,
)

df_assigned = va.assign_variants_to_regions(df_filt, uniprot_lookup, symbol_lookup)
df_assigned.head()

In [ ]:
# How many of your 781 regions got at least one variant?
region_coverage = df_assigned.groupby("region_id").size()
n_regions_with_variants = len(region_coverage)
print(f"Regions with at least one variant: {n_regions_with_variants} / 781")
print(f"Regions with no variants: {781 - n_regions_with_variants}")

print("\nVariants per region distribution:")
print(region_coverage.describe())

# Per-group coverage
print("\nVariants per group:")
print(df_assigned.groupby("group").size())



In [ ]:
##### FINAL ANNOTATIONS

In [ ]:
df_with_af = va.load_and_merge_frequencies(
    df_assigned,
    af_tsv_path="/mnt/d/phd/scripts/16_ev_signature_predictor/data/bcftools/combined_joint_variants_AF.tsv",
)

# Quick audit
print("\nAF coverage by group:")
print(df_with_af.groupby("group")["AF_joint"].apply(
    lambda s: f"{s.notna().sum()} / {len(s)} ({s.notna().mean():.1%})"
))
print("\nAF distribution:")
print(df_with_af["AF_joint"].describe())
print("\nRow with missing AF (sample):")
print(df_with_af[df_with_af["AF_joint"].isna()].head(3)[
    ["CHROM","POS","REF","ALT","SYMBOL","region_id","group"]
])

In [ ]:
df_with_af = va.parse_amino_acids_column(df_with_af)

# Sanity check across consequence types
sample_cons = ["missense_variant", "synonymous_variant", "stop_gained",
               "inframe_deletion", "inframe_insertion", "start_lost"]
for cons in sample_cons:
    subset = df_with_af[df_with_af["Consequence"] == cons]
    if len(subset) > 0:
        print(f"\n{cons}:")
        print(subset[["Amino_acids", "before_aa", "after_aa"]].head(3).to_string(index=False))

print(f"\nNulls in before_aa: {df_with_af['before_aa'].isna().sum()}")
print(f"Nulls in after_aa: {df_with_af['after_aa'].isna().sum()}")

In [ ]:


# Extract the UniProt IDs from our regions



with open("/mnt/d/phd/scripts/16_ev_signature_predictor/data/processed/genomic_coords_merged_win5_4groups.json") as f:
    regions = json.load(f)
uniprot_ids = sorted({r["protein"] for r in regions})

# ################# ONLY RUN THIS ONCE AND THEN LOAD THE FILE OTHERWISE YOU LOSE 1hr ##################
# # Load + filter AlphaMissense
# am = va.load_alphamissense_for_proteins(
#     am_tsv_path="/mnt/d/phd/scripts/16_ev_signature_predictor/data/alphamissense/AlphaMissense_aa_substitutions.tsv.gz",
#     uniprot_ids=uniprot_ids,
# )

# am.to_parquet(
#     "/mnt/d/phd/scripts/16_ev_signature_predictor/data/alphamissense/am_filtered_for_4_groups.parquet"
# )
# print("Saved.")
# print(am)
#######################################################################

am = pd.read_parquet(
    "/mnt/d/phd/scripts/16_ev_signature_predictor/data/alphamissense/am_filtered_for_4_groups.parquet"
)
print("Loaded:")
print(am)

df_final = va.merge_alphamissense(df_with_af, am)

# Missing protein check
missing = set(uniprot_ids) - set(am["uniprot_id"])
print(f"\nUniProt IDs missing from AlphaMissense: {missing}")

# Distribution check
missense = df_final[df_final["Consequence"].str.contains("missense_variant", na=False)]
print("\nPathogenicity by group (missense only):")
print(missense.groupby("group")["am_pathogenicity"].describe())

print("\nClass breakdown by group:")
print(missense.groupby(["group", "am_class"]).size().unstack(fill_value=0))

af_cols = [c for c in df_final.columns if c.startswith(("AC_", "AN_", "AF_"))]
print(f"Fixing {len(af_cols)} AF columns")

for col in af_cols:
    df_final[col] = pd.to_numeric(df_final[col], errors="coerce")

# Verify
print(df_final[af_cols].dtypes.value_counts())

In [ ]:
from pathlib import Path
from src.analysis_visualization.esm_llr import (
    download_esm_llr_zip,
    annotate_variants_with_esm,
)

ZIP_PATH = Path("data/esm1b/ALL_hum_isoforms_ESM1b_LLR.zip")

def add_esm_llr(df_final, zip_path=ZIP_PATH, force=False):
    """Add esm_llr to df_final. Idempotent unless force=True."""
    if "esm_llr" in df_final.columns and not force:
        print("esm_llr already present — skipping (force=True to redo)")
        return df_final

    download_esm_llr_zip(zip_path)
    df_final = annotate_variants_with_esm(
        df_final,
        zip_path=zip_path,
        uniprot_col="uniprot_accession",
        pos_col="Protein_position",
        after_col="after_aa",
        consequence_col="Consequence",
    )
    return df_final

df_final = add_esm_llr(df_final)

In [ ]:
####FILTERING STEP 

region_by_id = {r["region_id"]: r for r in regions}

def _check(row):
    region = region_by_id.get(row["region_id"])
    if region is None or pd.isna(row["protein_position_int"]):
        return None
    # Note: prot_region is 0-based half-open, so position offset is:
    pos = int(row["protein_position_int"]) - int(row["region_start_aa"]) - 1
    # The -1 accounts for protein_position_int being 1-based (VEP convention),
    # while region_start_aa is 0-based (our convention).
    if pos < 0 or pos >= len(region["prot_seq"]):
        return None
    if row["before_aa"] is None or row["before_aa"] == "-" or len(row["before_aa"]) != 1:
        return None
    return region["prot_seq"][pos] == row["before_aa"]

df_final["wt_match"] = df_final.apply(_check, axis=1)
print(df_final["wt_match"].value_counts(dropna=False))

df_final["rg_analysis_reliable"] = df_final["wt_match"] == True
df_for_rg = df_final[df_final["rg_analysis_reliable"]].copy()

print(f"\nTotal variants: {len(df_final):,}")
print(f"Usable for RG analysis: {len(df_for_rg):,}")
print(f"\nBreakdown by group:")
print(df_for_rg["group"].value_counts())

import src.analysis_visualization.rg_analysis as rga

df_for_rg = rga.compute_rg_disruption_columns(df_for_rg, region_by_id)


In [ ]:
from src.classifier.final_model import load_model, score_new_regions
from src.classifier.features_static import build_static_features

bundle = load_model("/mnt/d/phd/scripts/16_ev_signature_predictor/models/rg_classifier_final.joblib")

# build static features on the EXTENDED set — SAME config as training
X_ext = build_static_features(
    df_for_rg,
    region_by_id=region_by_id,
    include_wt_physchem=True,
    codon_source_aas=list("ADEGLPRS"),
    include_gc_codon_indices=True,
    codon_rates=None,
    physchem_delta_cache="/mnt/d/phd/scripts/16_ev_signature_predictor/data/processed/physchem_deltas_4groups.parquet",
)
print("X_ext:", X_ext.shape)

scores = score_new_regions(bundle, X_ext, return_proba=True)
print(scores.describe())

In [ ]:
import numpy as np
import pandas as pd
import shap

# ── 1) rebuild the exact matrix the RF scored on ────────────────────────────
rf        = bundle["rf"]
imp       = bundle["imputer"]
feat_cols = bundle["feature_cols"]            # 111, in model order

X_aligned = X_ext.reindex(columns=feat_cols)  # exact cols, exact order; missing->NaN
X_imp = imp.transform(X_aligned.values)        # TRAINING medians (not refit)

# sanity: these scores must match `scores` from score_new_regions
proba = rf.predict_proba(X_imp)[:, 1]
assert np.allclose(proba, scores.reindex(X_ext.index).values), "score mismatch — alignment drift"

# ── 2) SHAP on the scored matrix ────────────────────────────────────────────
explainer = shap.TreeExplainer(rf)
sv = explainer.shap_values(X_imp)
# RandomForest binary: shap_values may be a list [class0, class1] or a 3D array.
if isinstance(sv, list):
    sv_pos = sv[1]
elif sv.ndim == 3:
    sv_pos = sv[:, :, 1]
else:
    sv_pos = sv
sv_pos = pd.DataFrame(sv_pos, index=X_ext.index, columns=feat_cols)

# ── 3) group label per region (from region_by_id) ──────────────────────────
group_of = {rid: region_by_id[rid].get("group")
            for rid in X_ext.index if rid in region_by_id}

# ── 4) per-region top contributors up and down ──────────────────────────────
TOP_N = 3
def _fmt(region_row):
    s = region_row.sort_values()
    down = s.head(TOP_N)                         # most negative (push score down)
    up   = s.tail(TOP_N).iloc[::-1]              # most positive (push score up)
    up_str   = "; ".join(f"{k}={v:+.3f}" for k, v in up.items())
    down_str = "; ".join(f"{k}={v:+.3f}" for k, v in down.items())
    return up_str, down_str

records = []
for rid in X_ext.index:
    up_str, down_str = _fmt(sv_pos.loc[rid])
    records.append({
        "region_id": rid,
        "group": group_of.get(rid),
        "score": float(scores.loc[rid]),
        "top_features_up": up_str,
        "top_features_down": down_str,
    })

out = pd.DataFrame(records).sort_values(["group", "score"], ascending=[True, False])



In [ ]:
import re

def to_one_based(region_id: str) -> str:
    """Convert 'UNIPROTID_START_END' where START/END are 0-based Python
    slice coordinates (end-exclusive) into 1-based inclusive coordinates.

    e.g. 'Q6RI45_1678_1724' (0-based, slice-style)
      -> 'Q6RI45_1679_1724' (1-based, inclusive)
    """
    uniprot_id, start, end = region_id.rsplit("_", 2)
    start_1based = int(start) + 1
    end_1based = int(end)  # slice end (exclusive, 0-based) == inclusive end, 1-based
    return f"{uniprot_id}_{start_1based}_{end_1based}"



out["region_id_1based"] = out["region_id"].apply(to_one_based)

out = out.drop(columns=["region_id"])
out = out.rename(columns={"region_id_1based": "region_id"})

# move region_id to be the first column
cols = ["region_id"] + [c for c in out.columns if c != "region_id"]
out = out[cols]
out = out.set_index("region_id", drop=False) 

In [ ]:
out

In [ ]:
# ── 5) write CSV ─────────────────────────────────────────────────────────────
OUT_PATH = "/mnt/d/phd/scripts/16_ev_signature_predictor/data/output/predictions_4groups_inference.csv"
out.to_csv(OUT_PATH, sep=",", index=False)
print(f"wrote {len(out)} regions -> {OUT_PATH}")

# quick look: score distribution by group
print(out.groupby("group")["score"].describe()[["count","mean","50%","min","max"]])
print(out.head(10).to_string(index=False))